In [1]:
import pandas as pd
from pathlib import Path
import json

# Set your base path
base_path = Path("/mnt/data1/HCI WORK/Low Engagement Detection/Pupil Recordings/User_1_recording/000")
gaze_file = base_path / "exports/000/gaze_positions.csv"
pupil_file = base_path / "exports/000/pupil_positions.csv"

# Load gaze and pupil data
gaze_df = pd.read_csv(gaze_file)
pupil_df = pd.read_csv(pupil_file)

# Show basic info
print("✅ Loaded gaze_positions.csv")
display(gaze_df.head())

print("✅ Loaded pupil_positions.csv")
display(pupil_df.head())


✅ Loaded gaze_positions.csv


,gaze_timestamp,world_index,confidence,norm_pos_x,norm_pos_y,base_data,gaze_point_3d_x,gaze_point_3d_y,gaze_point_3d_z,eye_center0_3d_x,...,eye_center0_3d_z,gaze_normal0_x,gaze_normal0_y,gaze_normal0_z,eye_center1_3d_x,eye_center1_3d_y,eye_center1_3d_z,gaze_normal1_x,gaze_normal1_y,gaze_normal1_z
0,4.075473e+06,0,0.867574,0.457965,0.325192,4075473.150582-0 4075473.148068-1,-7.507259,14.204164,126.441752,23.130576,...,-10.007133,-0.218970,-0.093367,0.971254,-39.125153,19.13488,2.457676,0.246903,0.019387,0.968846
1,4.075473e+06,0,0.932637,0.459933,0.330367,4075473.150582-0 4075473.156137-1,-6.964969,13.341146,124.060080,23.130576,...,-10.007133,-0.218970,-0.093367,0.971254,-39.125153,19.13488,2.457676,0.255499,0.004974,0.966797
2,4.075473e+06,0,0.942398,0.459731,0.330441,4075473.158651-0 4075473.156137-1,-6.998550,13.319198,123.932370,23.130576,...,-10.007133,-0.219400,-0.093784,0.971117,-39.125153,19.13488,2.457676,0.255499,0.004974,0.966797
3,4.075473e+06,0,0.934192,0.460310,0.331050,4075473.158651-0 4075473.164206-1,-6.843251,13.175373,123.245231,23.130576,...,-10.007133,-0.219400,-0.093784,0.971117,-39.125153,19.13488,2.457676,0.258014,0.002325,0.966138
4,4.075473e+06,1,0.915042,0.460259,0.331035,4075473.166721-0 4075473.164206-1,-6.851752,13.173718,123.213336,23.130576,...,-10.007133,-0.219509,-0.093831,0.971088,-39.125153,19.13488,2.457676,0.258014,0.002325,0.966138


✅ Loaded pupil_positions.csv


,pupil_timestamp,world_index,eye_id,confidence,norm_pos_x,norm_pos_y,diameter,method,ellipse_center_x,ellipse_center_y,...,circle_3d_normal_y,circle_3d_normal_z,circle_3d_radius,theta,phi,projected_sphere_center_x,projected_sphere_center_y,projected_sphere_axis_a,projected_sphere_axis_b,projected_sphere_angle
0,4.075473e+06,0,1,0.856733,0.641544,0.580582,22.625506,pye3d 0.3.0 real-time,123.176382,80.528242,...,-0.158986,-0.924381,1.215996,1.730460,-1.929679,136.862456,91.174224,159.111360,159.111360,0.0
1,4.075473e+06,0,1,0.856733,0.641636,0.580408,22.633015,2d c++,123.194099,80.561676,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.075473e+06,0,0,0.878414,0.505241,0.358888,26.246832,2d c++,97.006195,123.093460,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.075473e+06,0,0,0.878414,0.505235,0.358890,26.246478,pye3d 0.3.0 real-time,97.005177,123.093162,...,0.255325,-0.773381,1.284701,1.312612,-2.214475,127.897854,94.475577,172.898536,172.898536,0.0
4,4.075473e+06,0,1,0.986860,0.641555,0.580174,22.840420,2d c++,123.178543,80.606583,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Step 1: Extract start_time_synced_s from JSON
import json

json_path = "/mnt/data1/HCI WORK/Low Engagement Detection/Pupil Recordings/User_1_recording/000/info.player.json"

with open(json_path, 'r') as f:
    player_info = json.load(f)

start_time_synced = player_info["start_time_synced_s"]
print(f"Start time (synced): {start_time_synced}")



Start time (synced): 4075473.136655425


In [3]:
# Step 2: Prepare and Normalize Timestamps
# Normalize gaze timestamps
gaze_df["timestamp_synced"] = gaze_df["gaze_timestamp"] - start_time_synced
gaze_df = gaze_df.sort_values("timestamp_synced").reset_index(drop=True)

pupil_df["timestamp_synced"] = pupil_df["pupil_timestamp"] - start_time_synced
pupil_df = pupil_df.sort_values("timestamp_synced").reset_index(drop=True)


In [4]:
gaze_df.shape

(595298, 22)

In [5]:
pupil_df.shape

(1190598, 35)

In [58]:
def estimate_framerate(df, timestamp_col):
    # Compute time differences between consecutive timestamps
    time_diffs = df[timestamp_col].sort_values().diff().dropna()
    avg_diff = time_diffs.mean()
    if avg_diff > 0:
        return 1 / avg_diff
    return None

gaze_fps = estimate_framerate(gaze_df, "gaze_timestamp")
pupil_fps = estimate_framerate(pupil_df, "pupil_timestamp")

print(f"🎥 Estimated Gaze Sampling Rate: {gaze_fps:.2f} Hz")
print(f"🎥 Estimated Pupil Sampling Rate: {pupil_fps:.2f} Hz")


🎥 Estimated Gaze Sampling Rate: 247.83 Hz
🎥 Estimated Pupil Sampling Rate: 495.65 Hz


In [59]:
# Round to millisecond precision to avoid tiny floating point mismatch
gaze_ts = gaze_df["gaze_timestamp"].round(3)
pupil_ts = pupil_df["pupil_timestamp"].round(3)

# Find common timestamps
common_ts = pd.Series(list(set(gaze_ts) & set(pupil_ts)))
num_common = len(common_ts)

print(f"🔎 Total gaze timestamps: {len(gaze_ts)}")
print(f"🔎 Total pupil timestamps: {len(pupil_ts)}")
print(f"🔁 Common (rounded) timestamps: {num_common}")

if num_common > 0:
    print("✅ Some timestamps match exactly (to 1ms).")
    display(common_ts.sort_values().head())
else:
    print("⚠️ No exact matches found — will need to align based on nearest timestamps.")


🔎 Total gaze timestamps: 595298
🔎 Total pupil timestamps: 1190598
🔁 Common (rounded) timestamps: 100191
✅ Some timestamps match exactly (to 1ms).


59128    4075474.601
20211    4075474.651
81408    4075474.665
59127    4075474.964
20212    4075474.966
dtype: float64

In [60]:
# Round to ms to ensure clean grouping
pupil_df["rounded_ts"] = pupil_df["pupil_timestamp"].round(3)

# Count how many rows per timestamp
ts_counts = pupil_df.groupby("rounded_ts").size().value_counts().sort_index()

print("Rows per unique pupil timestamp (rounded to ms):")
display(ts_counts)


Rows per unique pupil timestamp (rounded to ms):


2    595299
Name: count, dtype: int64

In [61]:
eye_counts = pupil_df.groupby(["rounded_ts", "eye_id"]).size().unstack(fill_value=0)
print("Sample eye_id distribution per timestamp:")
display(eye_counts.head(10))

# Count how many timestamps have both eye_id = 0 and 1
both_eyes = (eye_counts[0] > 0) & (eye_counts[1] > 0)
print(f"Number of timestamps with both eyes recorded: {both_eyes.sum()} out of {len(eye_counts)}")


Sample eye_id distribution per timestamp:


eye_id,0,1
rounded_ts,,
4075473.148,0,2
4075473.151,2,0
4075473.156,0,2
4075473.159,2,0
4075473.164,0,2
4075473.167,2,0
4075473.172,0,2
4075473.175,2,0
4075473.180,0,2


Number of timestamps with both eyes recorded: 0 out of 595299


In [62]:
print("Unique methods used in pupil_df:")
print(pupil_df["method"].value_counts())


Unique methods used in pupil_df:
method
pye3d 0.3.0 real-time    595299
2d c++                   595299
Name: count, dtype: int64


✅ Conclusion from Your Outputs
🔎 Sampling Structure:
595,299 unique timestamps, each with 2 rows ⟶ total of 1,190,598 rows, as expected.

Each timestamp has one row for each method:

One for pye3d 0.3.0 real-time (3D model)

One for 2d c++ (basic 2D detection)

Within those, each method seems to alternate between eye_id 0 and 1.

🧠 Therefore:
Each timestamp = 1 pupil sample per eye per method (2 methods × 2 eyes = 4 possible rows per timestamp).

But in your case, only 2 methods (each timestamp has just one eye per method), hence 2 rows per timestamp.



Start time (synced): 4075473.136655425


In [65]:
# Prepare pupil data for each method
pupil_pye3d = pupil_df[(pupil_df["method"].str.contains("pye3d")) & (pupil_df["eye_id"] == 0)].copy()
pupil_2dcpp = pupil_df[(pupil_df["method"].str.contains("2d c++")) & (pupil_df["eye_id"] == 0)].copy()

# Normalize timestamps
pupil_pye3d["timestamp_synced"] = pupil_pye3d["pupil_timestamp"] - start_time_synced
pupil_2dcpp["timestamp_synced"] = pupil_2dcpp["pupil_timestamp"] - start_time_synced

# Sort
pupil_pye3d = pupil_pye3d.sort_values("timestamp_synced").reset_index(drop=True)
pupil_2dcpp = pupil_2dcpp.sort_values("timestamp_synced").reset_index(drop=True)


In [66]:
# Step 3: Merge Using merge_asof (2ms tolerance)


# Merge with pye3d
merged_pye3d = pd.merge_asof(
    gaze_df,
    pupil_pye3d,
    on="timestamp_synced",
    direction="nearest",
    tolerance=0.002,
    suffixes=("_gaze", "_pupil_pye3d")
)

# Merge with 2d c++
merged_2dcpp = pd.merge_asof(
    gaze_df,
    pupil_2dcpp,
    on="timestamp_synced",
    direction="nearest",
    tolerance=0.002,
    suffixes=("_gaze", "_pupil_2dcpp")
)


In [67]:
print(f"🧪 Rows in merged_pye3d: {merged_pye3d.shape[0]}")
print(f"🧪 Rows in merged_2dcpp: {merged_2dcpp.shape[0]}")

# Optionally check how many rows got matched
print("✅ pye3d non-null merge rows:", merged_pye3d["pupil_timestamp"].notnull().sum())
print("✅ 2d c++ non-null merge rows:", merged_2dcpp["pupil_timestamp"].notnull().sum())


🧪 Rows in merged_pye3d: 595298
🧪 Rows in merged_2dcpp: 595298
✅ pye3d non-null merge rows: 291082
✅ 2d c++ non-null merge rows: 291082


In [68]:
merged_pye3d['eye_id'].head(10)

0    0.0
1    NaN
2    0.0
3    NaN
4    0.0
5    NaN
6    0.0
7    NaN
8    0.0
9    NaN
Name: eye_id, dtype: float64

In [69]:
merged_pye3d.isnull().sum()

gaze_timestamp                    0
world_index_gaze                  0
confidence_gaze                   0
norm_pos_x_gaze                   0
norm_pos_y_gaze                   0
base_data                         0
gaze_point_3d_x                   0
gaze_point_3d_y                   0
gaze_point_3d_z                   0
eye_center0_3d_x              49538
eye_center0_3d_y              49538
eye_center0_3d_z              49538
gaze_normal0_x                49538
gaze_normal0_y                49538
gaze_normal0_z                49538
eye_center1_3d_x              50631
eye_center1_3d_y              50631
eye_center1_3d_z              50631
gaze_normal1_x                50631
gaze_normal1_y                50631
gaze_normal1_z                50631
timestamp_synced                  0
pupil_timestamp              304216
world_index_pupil_pye3d      304216
eye_id                       304216
confidence_pupil_pye3d       304216
norm_pos_x_pupil_pye3d       304216
norm_pos_y_pupil_pye3d      

In [70]:
merged_2dcpp.isnull().sum()

gaze_timestamp                    0
world_index_gaze                  0
confidence_gaze                   0
norm_pos_x_gaze                   0
norm_pos_y_gaze                   0
base_data                         0
gaze_point_3d_x                   0
gaze_point_3d_y                   0
gaze_point_3d_z                   0
eye_center0_3d_x              49538
eye_center0_3d_y              49538
eye_center0_3d_z              49538
gaze_normal0_x                49538
gaze_normal0_y                49538
gaze_normal0_z                49538
eye_center1_3d_x              50631
eye_center1_3d_y              50631
eye_center1_3d_z              50631
gaze_normal1_x                50631
gaze_normal1_y                50631
gaze_normal1_z                50631
timestamp_synced                  0
pupil_timestamp              304216
world_index_pupil_2dcpp      304216
eye_id                       304216
confidence_pupil_2dcpp       304216
norm_pos_x_pupil_2dcpp       304216
norm_pos_y_pupil_2dcpp      

In [71]:
merged_2dcpp.columns

Index(['gaze_timestamp', 'world_index_gaze', 'confidence_gaze',
       'norm_pos_x_gaze', 'norm_pos_y_gaze', 'base_data', 'gaze_point_3d_x',
       'gaze_point_3d_y', 'gaze_point_3d_z', 'eye_center0_3d_x',
       'eye_center0_3d_y', 'eye_center0_3d_z', 'gaze_normal0_x',
       'gaze_normal0_y', 'gaze_normal0_z', 'eye_center1_3d_x',
       'eye_center1_3d_y', 'eye_center1_3d_z', 'gaze_normal1_x',
       'gaze_normal1_y', 'gaze_normal1_z', 'timestamp_synced',
       'pupil_timestamp', 'world_index_pupil_2dcpp', 'eye_id',
       'confidence_pupil_2dcpp', 'norm_pos_x_pupil_2dcpp',
       'norm_pos_y_pupil_2dcpp', 'diameter', 'method', 'ellipse_center_x',
       'ellipse_center_y', 'ellipse_axis_a', 'ellipse_axis_b', 'ellipse_angle',
       'diameter_3d', 'model_confidence', 'model_id', 'sphere_center_x',
       'sphere_center_y', 'sphere_center_z', 'sphere_radius',
       'circle_3d_center_x', 'circle_3d_center_y', 'circle_3d_center_z',
       'circle_3d_normal_x', 'circle_3d_normal_y', 'c

# Cleaning Merged-eye data

In [72]:
# Columns for 2dcpp (has 'confidence_pupil_2dcpp')
columns_2dcpp = [
    'timestamp_synced', 'eye_id',
    'diameter', 'confidence_pupil_2dcpp',
    'norm_pos_x_gaze', 'norm_pos_y_gaze', 'confidence_gaze',
    'gaze_normal0_x', 'gaze_normal0_y', 'gaze_normal0_z',
    'gaze_point_3d_x', 'gaze_point_3d_y', 'gaze_point_3d_z',
    'ellipse_center_x', 'ellipse_center_y',
    'ellipse_axis_a', 'ellipse_axis_b', 'ellipse_angle',
    'diameter_3d', 'projected_sphere_axis_a', 'projected_sphere_axis_b'
]

# Columns for pye3d (use 'confidence_gaze' only — pupil confidence not available)
columns_pye3d = [
    'timestamp_synced', 'eye_id',
    'diameter',  # still useful from pye3d
    'norm_pos_x_gaze', 'norm_pos_y_gaze', 'confidence_gaze',
    'gaze_normal0_x', 'gaze_normal0_y', 'gaze_normal0_z',
    'gaze_point_3d_x', 'gaze_point_3d_y', 'gaze_point_3d_z',
    'ellipse_center_x', 'ellipse_center_y',
    'ellipse_axis_a', 'ellipse_axis_b', 'ellipse_angle',
    'diameter_3d', 'projected_sphere_axis_a', 'projected_sphere_axis_b'
]

# Subset cleanly
merged_pye3d_clean = merged_pye3d[columns_pye3d].copy()
merged_2dcpp_clean = merged_2dcpp[columns_2dcpp].copy()


In [73]:
merged_2dcpp_clean.head()

,timestamp_synced,eye_id,diameter,confidence_pupil_2dcpp,norm_pos_x_gaze,norm_pos_y_gaze,confidence_gaze,gaze_normal0_x,gaze_normal0_y,gaze_normal0_z,...,gaze_point_3d_y,gaze_point_3d_z,ellipse_center_x,ellipse_center_y,ellipse_axis_a,ellipse_axis_b,ellipse_angle,diameter_3d,projected_sphere_axis_a,projected_sphere_axis_b
0,0.012670,0.0,26.246832,0.878414,0.457965,0.325192,0.867574,-0.218970,-0.093367,0.971254,...,14.204164,126.441752,97.006195,123.093460,21.167027,26.246832,150.472961,NaN,NaN,NaN
1,0.016704,NaN,NaN,NaN,0.459933,0.330367,0.932637,-0.218970,-0.093367,0.971254,...,13.341146,124.060080,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.020739,0.0,26.234453,0.897935,0.459731,0.330441,0.942398,-0.219400,-0.093784,0.971117,...,13.319198,123.932370,96.989586,123.072639,21.199324,26.234453,149.264557,NaN,NaN,NaN
3,0.024773,NaN,NaN,NaN,0.460310,0.331050,0.934192,-0.219400,-0.093784,0.971117,...,13.175373,123.245231,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.028808,0.0,26.328188,0.859636,0.460259,0.331035,0.915042,-0.219509,-0.093831,0.971088,...,13.173718,123.213336,97.023170,123.031990,21.158894,26.328188,149.669983,NaN,NaN,NaN


In [74]:
# Check Null Percentages
# Now let’s see which columns have too many missing values (>50%) and drop them.


# Check null percentage for pye3d
nulls_pye3d = merged_pye3d_clean.isnull().mean().sort_values(ascending=False)
print("🧪 Null % - Pye3d")
print(nulls_pye3d)

# Check null percentage for 2dcpp
nulls_2dcpp = merged_2dcpp_clean.isnull().mean().sort_values(ascending=False)
print("\n🧪 Null % - 2d-c++")
print(nulls_2dcpp)

# Optional: drop columns with > 50% nulls
# merged_pye3d_clean = merged_pye3d_clean.loc[:, nulls_pye3d <= 0.5]
# merged_2dcpp_clean = merged_2dcpp_clean.loc[:, nulls_2dcpp <= 0.5]



🧪 Null % - Pye3d
eye_id                     0.511031
diameter                   0.511031
projected_sphere_axis_a    0.511031
projected_sphere_axis_b    0.511031
ellipse_axis_b             0.511031
ellipse_axis_a             0.511031
diameter_3d                0.511031
ellipse_angle              0.511031
ellipse_center_y           0.511031
ellipse_center_x           0.511031
gaze_normal0_x             0.083215
gaze_normal0_z             0.083215
gaze_normal0_y             0.083215
timestamp_synced           0.000000
norm_pos_x_gaze            0.000000
gaze_point_3d_z            0.000000
confidence_gaze            0.000000
gaze_point_3d_y            0.000000
gaze_point_3d_x            0.000000
norm_pos_y_gaze            0.000000
dtype: float64

🧪 Null % - 2d-c++
projected_sphere_axis_b    1.000000
diameter_3d                1.000000
projected_sphere_axis_a    1.000000
confidence_pupil_2dcpp     0.511031
eye_id                     0.511031
ellipse_center_x           0.511031
ellipse_cente

In [75]:
# # Add time_window for merging later
# merged_pye3d_clean['time_window'] = (merged_pye3d_clean['timestamp_synced'] // 5).astype(int) * 5
# merged_2dcpp_clean['time_window'] = (merged_2dcpp_clean['timestamp_synced'] // 5).astype(int) * 5


In [76]:
# Drop rows with null eye_id
merged_pye3d_clean_drop = merged_pye3d_clean[merged_pye3d_clean['eye_id'].notnull()].copy()
merged_2dcpp_clean_drop= merged_2dcpp_clean[merged_2dcpp_clean['eye_id'].notnull()].copy()

print(f"✅ Rows after dropping null eye_id:")
print(f"pye3d → {merged_pye3d_clean_drop.shape}")
print(f"2dcpp → {merged_2dcpp_clean_drop.shape}")


✅ Rows after dropping null eye_id:
pye3d → (291082, 20)
2dcpp → (291082, 21)


In [77]:
merged_pye3d_clean_drop.describe()

,timestamp_synced,eye_id,diameter,norm_pos_x_gaze,norm_pos_y_gaze,confidence_gaze,gaze_normal0_x,gaze_normal0_y,gaze_normal0_z,gaze_point_3d_x,gaze_point_3d_y,gaze_point_3d_z,ellipse_center_x,ellipse_center_y,ellipse_axis_a,ellipse_axis_b,ellipse_angle,diameter_3d,projected_sphere_axis_a,projected_sphere_axis_b
count,291082.000000,291082.0,2.910820e+05,291082.000000,291082.000000,291082.000000,291082.000000,291082.000000,291082.000000,291082.000000,291082.000000,2.910820e+05,291082.000000,291082.000000,291082.000000,2.910820e+05,291082.000000,291082.000000,291082.000000,291082.000000
mean,1210.338960,0.0,1.310707e+06,0.580501,0.455111,0.846964,0.072233,-0.146626,0.921127,32.543676,-24.470832,2.684041e+02,104.076998,125.559369,18.163471,1.310707e+06,124.122479,2.695712,144.394039,144.394039
std,698.822506,0.0,3.624511e+08,0.778006,1.148845,0.222678,0.266217,0.184848,0.140597,898.051031,2093.360293,4.976645e+03,13.142680,13.792227,3.940753,3.624511e+08,29.533700,0.489335,9.776633,9.776633
min,0.012670,0.0,0.000000e+00,-24.975807,-100.000000,0.000000,-0.988526,-0.999790,-0.833450,-101086.937733,-638073.477093,8.647185e-03,-32.000000,-160.000000,0.000000,0.000000e+00,0.000000,-0.175075,114.943725,114.943725
25%,587.744491,0.0,2.056152e+01,0.486867,0.410307,0.851303,-0.125624,-0.240147,0.940255,-1.695494,-19.632409,1.160762e+02,95.784176,125.381806,16.480282,2.056152e+01,115.445324,2.459832,138.728505,138.728505
50%,1229.510568,0.0,2.200190e+01,0.567615,0.506078,0.928505,0.020360,-0.166096,0.964436,17.221435,-6.680781,1.327105e+02,102.328115,128.774714,18.476197,2.200190e+01,128.934218,2.711237,145.301725,145.301725
75%,1815.190928,0.0,2.379707e+01,0.670352,0.597741,0.984118,0.200341,-0.095644,0.979482,43.703232,4.589987,1.656254e+02,112.454058,131.421082,20.311041,2.379707e+01,140.026902,2.962433,151.705672,151.705672
max,2402.089755,0.0,1.332708e+11,100.000000,55.633273,1.000000,0.999369,0.999824,0.999998,264626.267553,45695.826936,1.254894e+06,194.370326,1632.000000,386.948555,1.332708e+11,180.000000,6.125452,172.898536,172.898536


In [78]:
merged_pye3d_clean_drop.head(10)

,timestamp_synced,eye_id,diameter,norm_pos_x_gaze,norm_pos_y_gaze,confidence_gaze,gaze_normal0_x,gaze_normal0_y,gaze_normal0_z,gaze_point_3d_x,gaze_point_3d_y,gaze_point_3d_z,ellipse_center_x,ellipse_center_y,ellipse_axis_a,ellipse_axis_b,ellipse_angle,diameter_3d,projected_sphere_axis_a,projected_sphere_axis_b
0,0.012670,0.0,26.246478,0.457965,0.325192,0.867574,-0.218970,-0.093367,0.971254,-7.507259,14.204164,126.441752,97.005177,123.093162,21.223173,26.246478,150.332112,2.569402,172.898536,172.898536
2,0.020739,0.0,26.236391,0.459731,0.330441,0.942398,-0.219400,-0.093784,0.971117,-6.998550,13.319198,123.932370,96.992030,123.077674,21.207424,26.236391,150.332234,2.568335,172.898536,172.898536
4,0.028808,0.0,26.329182,0.460259,0.331035,0.915042,-0.219509,-0.093831,0.971088,-6.851752,13.173718,123.213336,97.023265,123.035945,21.283171,26.329182,150.337882,2.577224,172.898536,172.898536
6,0.036877,0.0,26.185488,0.463897,0.335801,0.896547,-0.213082,-0.097803,0.972127,-6.176477,12.744445,124.344770,97.156343,123.021972,21.207275,26.185488,149.739198,2.563100,172.898536,172.898536
8,0.044946,0.0,26.040883,0.465814,0.336097,0.920271,-0.210267,-0.098382,0.972681,-5.794357,12.720168,124.455831,97.029988,123.072615,21.107507,26.040883,149.570193,2.549850,172.898536,172.898536
10,0.053016,0.0,26.236564,0.463473,0.341075,0.919888,-0.211940,-0.097060,0.972451,-6.334301,12.286535,125.835317,96.970694,122.934028,21.260780,26.236564,149.777982,2.568946,172.898536,172.898536
12,0.061085,0.0,26.307008,0.463807,0.341089,0.881762,-0.211892,-0.098073,0.972360,-6.247712,12.248714,125.468087,96.983545,123.003199,21.308634,26.307008,149.694603,2.575866,172.898536,172.898536
14,0.069154,0.0,26.207166,0.463095,0.347064,0.891370,-0.211512,-0.096938,0.972556,-6.450344,11.669263,126.654691,96.979245,122.946841,21.242233,26.207166,149.760038,2.566118,172.898536,172.898536
16,0.077223,0.0,26.359192,0.463261,0.347377,0.961301,-0.212833,-0.095365,0.972424,-6.346365,11.506475,125.280455,96.967316,122.913635,21.364374,26.359192,149.936902,2.581002,172.898536,172.898536
18,0.085292,0.0,26.246472,0.461353,0.344154,0.891355,-0.212559,-0.095795,0.972441,-6.878094,12.129646,127.919501,96.984120,122.962306,21.272047,26.246472,149.882704,2.569945,172.898536,172.898536


In [79]:
# Null Percentage
def null_percentage_report(df, name="DataFrame"):
    null_percent = df.isnull().mean().sort_values(ascending=False) * 100
    print(f"\n📊 Null Percentage Report for {name}:\n")
    display(null_percent.to_frame(name='Null %'))

    # Optional: Drop columns > 50% null
    cols_to_drop = null_percent[null_percent > 50].index.tolist()
    print(f"\n🗑️ Columns to drop in {name} (null % > 50): {cols_to_drop}")
    return cols_to_drop


In [80]:
# Step 2: Run null analysis
cols_drop_pye3d = null_percentage_report(merged_pye3d_clean_drop, "pye3d")
cols_drop_2dcpp = null_percentage_report(merged_2dcpp_clean_drop, "2dcpp")



📊 Null Percentage Report for pye3d:



,Null %
timestamp_synced,0.0
eye_id,0.0
diameter,0.0
norm_pos_x_gaze,0.0
norm_pos_y_gaze,0.0
confidence_gaze,0.0
gaze_normal0_x,0.0
gaze_normal0_y,0.0
gaze_normal0_z,0.0
gaze_point_3d_x,0.0



🗑️ Columns to drop in pye3d (null % > 50): []

📊 Null Percentage Report for 2dcpp:



,Null %
projected_sphere_axis_b,100.0
diameter_3d,100.0
projected_sphere_axis_a,100.0
eye_id,0.0
timestamp_synced,0.0
norm_pos_x_gaze,0.0
confidence_pupil_2dcpp,0.0
diameter,0.0
norm_pos_y_gaze,0.0
gaze_normal0_z,0.0



🗑️ Columns to drop in 2dcpp (null % > 50): ['projected_sphere_axis_b', 'diameter_3d', 'projected_sphere_axis_a']


In [81]:
# Convert timestamp to seconds (if not already), then bin into 5s windows
merged_pye3d_clean['window_id'] = (merged_pye3d_clean['timestamp_synced'] // 5).astype(int)
merged_2dcpp_clean['window_id'] = (merged_2dcpp_clean['timestamp_synced'] // 5).astype(int)


In [92]:
merged_2dcpp_clean['window_id'].value_counts().sort_index().head()


window_id
0    1237
1    1239
2    1239
3    1239
4    1231
Name: count, dtype: int64

In [83]:
merged_2dcpp_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 595298 entries, 0 to 595297
Data columns (total 22 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   timestamp_synced         595298 non-null  float64
 1   eye_id                   291082 non-null  float64
 2   diameter                 291082 non-null  float64
 3   confidence_pupil_2dcpp   291082 non-null  float64
 4   norm_pos_x_gaze          595298 non-null  float64
 5   norm_pos_y_gaze          595298 non-null  float64
 6   confidence_gaze          595298 non-null  float64
 7   gaze_normal0_x           545760 non-null  float64
 8   gaze_normal0_y           545760 non-null  float64
 9   gaze_normal0_z           545760 non-null  float64
 10  gaze_point_3d_x          595298 non-null  float64
 11  gaze_point_3d_y          595298 non-null  float64
 12  gaze_point_3d_z          595298 non-null  float64
 13  ellipse_center_x         291082 non-null  float64
 14  elli

In [84]:
merged_2dcpp_clean.head()

,timestamp_synced,eye_id,diameter,confidence_pupil_2dcpp,norm_pos_x_gaze,norm_pos_y_gaze,confidence_gaze,gaze_normal0_x,gaze_normal0_y,gaze_normal0_z,...,gaze_point_3d_z,ellipse_center_x,ellipse_center_y,ellipse_axis_a,ellipse_axis_b,ellipse_angle,diameter_3d,projected_sphere_axis_a,projected_sphere_axis_b,window_id
0,0.012670,0.0,26.246832,0.878414,0.457965,0.325192,0.867574,-0.218970,-0.093367,0.971254,...,126.441752,97.006195,123.093460,21.167027,26.246832,150.472961,NaN,NaN,NaN,0
1,0.016704,NaN,NaN,NaN,0.459933,0.330367,0.932637,-0.218970,-0.093367,0.971254,...,124.060080,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,0.020739,0.0,26.234453,0.897935,0.459731,0.330441,0.942398,-0.219400,-0.093784,0.971117,...,123.932370,96.989586,123.072639,21.199324,26.234453,149.264557,NaN,NaN,NaN,0
3,0.024773,NaN,NaN,NaN,0.460310,0.331050,0.934192,-0.219400,-0.093784,0.971117,...,123.245231,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,0.028808,0.0,26.328188,0.859636,0.460259,0.331035,0.915042,-0.219509,-0.093831,0.971088,...,123.213336,97.023170,123.031990,21.158894,26.328188,149.669983,NaN,NaN,NaN,0


In [85]:
agg_funcs = {
    'diameter': ['mean', 'std', 'min', 'max'],
    'confidence_gaze': ['mean', lambda x: (x < 0.6).sum()],
    'norm_pos_x_gaze': ['mean', 'std'],
    'norm_pos_y_gaze': ['mean', 'std'],
    'gaze_normal0_x': ['mean', 'std'],
    'gaze_normal0_y': ['mean', 'std'],
    'gaze_normal0_z': ['mean', 'std'],
    'gaze_point_3d_x': ['mean', 'std'],
    'gaze_point_3d_y': ['mean', 'std'],
    'gaze_point_3d_z': ['mean', 'std'],
    'ellipse_axis_a': ['mean'],
    'ellipse_axis_b': ['mean']
}



In [86]:
# Aggregate separately for pye3d and 2dcpp
eye_stats_pye3d = merged_pye3d_clean.groupby('window_id').agg(agg_funcs)
eye_stats_2dcpp = merged_2dcpp_clean.groupby('window_id').agg(agg_funcs)


In [87]:
# Flatten MultiIndex column names
eye_stats_pye3d.columns = ['_'.join([col[0], col[1]] if isinstance(col, tuple) else [col]) for col in eye_stats_pye3d.columns]
eye_stats_2dcpp.columns = ['_'.join([col[0], col[1]] if isinstance(col, tuple) else [col]) for col in eye_stats_2dcpp.columns]


In [88]:
eye_stats_pye3d.info()

<class 'pandas.core.frame.DataFrame'>
Index: 481 entries, 0 to 480
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   diameter_mean               481 non-null    float64
 1   diameter_std                481 non-null    float64
 2   diameter_min                481 non-null    float64
 3   diameter_max                481 non-null    float64
 4   confidence_gaze_mean        481 non-null    float64
 5   confidence_gaze_<lambda_0>  481 non-null    int64  
 6   norm_pos_x_gaze_mean        481 non-null    float64
 7   norm_pos_x_gaze_std         481 non-null    float64
 8   norm_pos_y_gaze_mean        481 non-null    float64
 9   norm_pos_y_gaze_std         481 non-null    float64
 10  gaze_normal0_x_mean         481 non-null    float64
 11  gaze_normal0_x_std          481 non-null    float64
 12  gaze_normal0_y_mean         481 non-null    float64
 13  gaze_normal0_y_std          481 non-null

In [89]:
import pandas as pd

physio_annot_path = "/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/alluser_normalized_allfeatures.csv"
full_physio_df = pd.read_csv(physio_annot_path)

# Filter only rows where p_id == 1
physio_df = full_physio_df[full_physio_df['P_id'] == 1]

# Preview and confirm structure
physio_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 253 entries, 0 to 252
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   GSR_mean             253 non-null    float64
 1   GSR_variance         253 non-null    float64
 2   HR_mean              253 non-null    float64
 3   HR_variance          253 non-null    float64
 4   75_percentile_GSR    253 non-null    float64
 5   75_percentile_HR     253 non-null    float64
 6   GSRmean_persen_diff  253 non-null    float64
 7   HRmean_persent_diff  253 non-null    float64
 8   GSRmean_diff         253 non-null    float64
 9   HRmean_diff          253 non-null    float64
 10  valence_acc_video    253 non-null    float64
 11  arousal_acc_video    253 non-null    float64
 12  P_id                 253 non-null    float64
 13  video_id             253 non-null    float64
 14  Score                245 non-null    float64
 15  time                 253 non-null    float64


In [90]:
physio_df.head()


,GSR_mean,GSR_variance,HR_mean,HR_variance,75_percentile_GSR,75_percentile_HR,GSRmean_persen_diff,HRmean_persent_diff,GSRmean_diff,HRmean_diff,...,video_id,Score,time,valence,arousal,videoID,start_time,end_time,probe,prev_window
0,0.424167,0.008937,0.3040,0.035968,0.500000,0.32,0.424167,0.691539,0.424167,0.691539,...,1.0,5.591294e-05,1.751711e+12,5.000000,5.00000,1.0,1.751711e+12,1.751711e+12,0,1
1,0.419167,0.008778,0.3072,0.039580,0.500000,0.32,0.419167,0.688339,0.419167,0.688339,...,1.0,8.733022e-10,1.751711e+12,6.333333,5.80000,1.0,1.751711e+12,1.751711e+12,0,0
2,0.433333,0.008750,0.3112,0.038739,0.500000,0.35,0.433333,0.684339,0.433333,0.684339,...,1.0,4.765301e-02,1.751711e+12,7.433750,7.56875,1.0,1.751711e+12,1.751711e+12,0,0
3,0.416667,0.007153,0.4064,0.052311,0.489583,0.56,0.416667,0.589139,0.416667,0.589139,...,1.0,6.480804e-02,1.751711e+12,7.433750,7.56875,1.0,1.751711e+12,1.751711e+12,0,0
4,0.411667,0.008447,0.3088,0.039875,0.500000,0.35,0.411667,0.686739,0.411667,0.686739,...,1.0,1.721657e-09,1.751711e+12,7.433750,7.56875,1.0,1.751711e+12,1.751711e+12,0,0


In [94]:
print(physio_df.columns)
print(eye_stats_pye3d.columns)


Index(['GSR_mean', 'GSR_variance', 'HR_mean', 'HR_variance',
       '75_percentile_GSR', '75_percentile_HR', 'GSRmean_persen_diff',
       'HRmean_persent_diff', 'GSRmean_diff', 'HRmean_diff',
       'valence_acc_video', 'arousal_acc_video', 'P_id', 'video_id', 'Score',
       'time', 'valence', 'arousal', 'videoID', 'start_time', 'end_time',
       'probe', 'prev_window'],
      dtype='object')
Index(['diameter_mean', 'diameter_std', 'diameter_min', 'diameter_max',
       'confidence_gaze_mean', 'confidence_gaze_<lambda_0>',
       'norm_pos_x_gaze_mean', 'norm_pos_x_gaze_std', 'norm_pos_y_gaze_mean',
       'norm_pos_y_gaze_std', 'gaze_normal0_x_mean', 'gaze_normal0_x_std',
       'gaze_normal0_y_mean', 'gaze_normal0_y_std', 'gaze_normal0_z_mean',
       'gaze_normal0_z_std', 'gaze_point_3d_x_mean', 'gaze_point_3d_x_std',
       'gaze_point_3d_y_mean', 'gaze_point_3d_y_std', 'gaze_point_3d_z_mean',
       'gaze_point_3d_z_std', 'ellipse_axis_a_mean', 'ellipse_axis_b_mean'],
      dty

In [91]:
physio_df = physio_df.rename(columns={'Unnamed: 0': 'window_id'})
final_merged = pd.merge(physio_df, eye_stats_pye3d, on='window_id', how='inner')


KeyError: 'window_id'

In [ ]:
print(final_merged.shape)
print(final_merged['window_id'].nunique())
final_merged.head(40)


In [ ]:
final_merged.to_csv("user_1_final_merged_eye_physio.csv", index=False)


In [6]:
import pandas as pd

physio_df = pd.read_csv("/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/alluser_normalized_allfeatures.csv")
physio_df = physio_df[physio_df["P_id"] == 1].copy()
physio_df["start_time_sec"] = physio_df["start_time"] / 1000
physio_df["end_time_sec"] = physio_df["end_time"] / 1000

print(physio_df[['start_time', 'end_time', 'start_time_sec', 'end_time_sec']].head())


     start_time      end_time  start_time_sec  end_time_sec
0  1.751711e+12  1.751711e+12    1.751711e+09  1.751711e+09
1  1.751711e+12  1.751711e+12    1.751711e+09  1.751711e+09
2  1.751711e+12  1.751711e+12    1.751711e+09  1.751711e+09
3  1.751711e+12  1.751711e+12    1.751711e+09  1.751711e+09
4  1.751711e+12  1.751711e+12    1.751711e+09  1.751711e+09
